In [1]:
# MBTI labels are kept in the same order used by the one-hot encodings.
mbti_types = [
    "INTJ","INTP","ENTJ","ENTP",
    "INFJ","INFP","ENFJ","ENFP",
    "ISTJ","ISFJ","ESTJ","ESFJ",
    "ISTP","ISFP","ESTP","ESFP"
]

In [2]:
# Total number of samples drawn for the generated training dataset.
DATASET_SIZE = 640
# Size produced by the sentence-transformer model.
EMBEDDING_DIM = 384

In [3]:
# Size of the compressed representation used by the neural networks.
LATENT_DIM = 64
# Number of values in each one-hot MBTI label.
LABEL_DIM = 16

In [4]:
# Number of MBTI types predicted by the classifier.
NUM_CLASSES = 16

In [5]:
# Number of samples processed in one training step.
BATCH_SIZE = 32

In [6]:
# Number of complete passes through the generated dataset.
NUM_EPOCHS = 5

In [7]:
"""Expand seed MBTI sentences into the CSV used by the training pipeline."""

'Expand seed MBTI sentences into the CSV used by the training pipeline.'

In [8]:
import csv
import random
import pandas as pd

In [9]:
mbti_df = pd.read_csv("mbti_texts_seeds.csv")
# Group seed sentences by type so each output row keeps its original label.
mbti_samples = {
    "INTJ": mbti_df[mbti_df['Type'] == 'INTJ']['Text'].to_list(),
    "INTP": mbti_df[mbti_df['Type'] == 'INTP']['Text'].to_list(),
    "ENTJ": mbti_df[mbti_df['Type'] == 'ENTJ']['Text'].to_list(),
    "ENTP": mbti_df[mbti_df['Type'] == 'ENTP']['Text'].to_list(),
    "INFJ": mbti_df[mbti_df['Type'] == 'INFJ']['Text'].to_list(),
    "INFP": mbti_df[mbti_df['Type'] == 'INFP']['Text'].to_list(),
    "ENFJ": mbti_df[mbti_df['Type'] == 'ENFJ']['Text'].to_list(),
    "ENFP": mbti_df[mbti_df['Type'] == 'ENFP']['Text'].to_list(),
    "ISTJ": mbti_df[mbti_df['Type'] == 'ISTJ']['Text'].to_list(),
    "ISFJ": mbti_df[mbti_df['Type'] == 'ISFJ']['Text'].to_list(),
    "ESTJ": mbti_df[mbti_df['Type'] == 'ESTJ']['Text'].to_list(),
    "ESFJ": mbti_df[mbti_df['Type'] == 'ESFJ']['Text'].to_list(),
    "ISTP": mbti_df[mbti_df['Type'] == 'ISTP']['Text'].to_list(),
    "ISFP": mbti_df[mbti_df['Type'] == 'ISFP']['Text'].to_list(),
    "ESTP": mbti_df[mbti_df['Type'] == 'ESTP']['Text'].to_list(),
    "ESFP": mbti_df[mbti_df['Type'] == 'ESFP']['Text'].to_list()
}

In [10]:
# Expand each list with simple suffix variations and write the generated CSV.
with open("mbti_texts.csv", "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["Type","Text"])
    for mbti, texts in mbti_samples.items():
        count = 0
        while count < 40:
            t = random.choice(texts)[:-1] + " " + random.choice(["always.", "often.", "sometimes.", "in my life.", "in my experience.", "in my opinion.", "in my perspective.", "in my view.", "in my understanding.", "in my belief.", "in my judgment.", "in my estimation.", "in my assessment.", "in my evaluation.", "in my analysis.", "in my interpretation.", "in my conclusion.", "in my reasoning.", "in my logic.", "in my deduction."])
            count += 1
            writer.writerow([mbti, t])

In [11]:
print("✅ MBTI text generation complete  |  wrote mbti_texts.csv")

✅ MBTI text generation complete  |  wrote mbti_texts.csv


In [12]:
"""Dataset helpers for sampled MBTI text and its prepared model inputs."""

'Dataset helpers for sampled MBTI text and its prepared model inputs.'

In [13]:
import torch
from torch.utils.data import Dataset
import pandas as pd

In [14]:
from constants import DATASET_SIZE, mbti_types

In [15]:
# The training dataset is sampled from the generated text collection.
mbtis_df = pd.read_csv("mbti_texts.csv")

In [16]:
class MBTIDataset(Dataset):
    """Hold sampled text labels and their later-generated model inputs."""

    def __init__(self, size=DATASET_SIZE):
        """Sample ``size`` text-label pairs and index their MBTI labels."""
        self.data = []
        self.sentence_embeddings = []
        self.label_one_hots = []
        for _ in range(size):
            row = mbtis_df.sample(n=1).iloc[0]
            text = row['Text']
            label = row['Type']
            self.data.append((text, label))
        self.label2idx = {t:i for i,t in enumerate(mbti_types)}

    def __len__(self):
        """Return the number of sampled records in the dataset."""
        return len(self.data)

    def __getitem__(self, idx):
        """Return the prepared embedding and one-hot label at ``idx``."""
        label_onehot = self.label_one_hots[idx]
        # Embeddings and labels are attached after the dataset is sampled.
        embedding = self.sentence_embeddings[idx]
        return embedding, label_onehot

In [17]:
# Trait descriptions are looked up after the classifier predicts an MBTI type.
traits_df = pd.read_csv("mbti_traits.csv")

In [18]:
"""Neural-network building blocks used by the MBTI training pipeline."""

'Neural-network building blocks used by the MBTI training pipeline.'

In [19]:
import torch
import torch.nn as nn
import torch.optim as optim

In [20]:
class Encoder(nn.Module):
    """Compress a sentence embedding into a smaller latent representation."""

    def __init__(self, input_dim, latent_dim):
        """Create the encoder for the configured input and latent dimensions."""
        super(Encoder, self).__init__()
        self.fc = nn.Sequential(
            nn.Linear(input_dim, 512),
            nn.ReLU(),
            nn.Linear(512, latent_dim)
        )

    def forward(self, x):
        """Return the latent representation for the input embeddings."""
        return self.fc(x)

In [21]:
class Decoder(nn.Module):
    """Reconstruct an embedding while conditioning on an MBTI label."""

    def __init__(self, latent_dim, label_dim, output_dim):
        """Create the decoder for latent vectors, labels, and output embeddings."""
        super(Decoder, self).__init__()
        self.fc = nn.Sequential(
            nn.Linear(latent_dim + label_dim, 512),
            nn.ReLU(),
            nn.Linear(512, output_dim),
            nn.Sigmoid()
        )

    def forward(self, z, labels):
        """Reconstruct an embedding from a latent vector and one-hot label."""
        return self.fc(torch.cat([z, labels], dim=1))

In [22]:
class Discriminator(nn.Module):
    """Estimate whether a latent vector resembles a real latent sample."""

    def __init__(self, latent_dim):
        """Create the discriminator for the configured latent dimension."""
        super(Discriminator, self).__init__()
        self.fc = nn.Sequential(
            nn.Linear(latent_dim, 256),
            nn.ReLU(),
            nn.Linear(256, 1),
            nn.Sigmoid()
        )

    def forward(self, z):
        """Return the discriminator probability for each latent vector."""
        return self.fc(z)

In [23]:
class Classifier(nn.Module):
    """Map a latent representation to logits for the 16 MBTI classes."""

    def __init__(self, latent_dim, num_classes=16):
        """Create the classifier for the latent size and number of classes."""
        super(Classifier, self).__init__()
        self.fc = nn.Linear(latent_dim, num_classes)

    def forward(self, z):
        """Return one classification logit for each configured MBTI class."""
        return self.fc(z)

In [24]:
"""Create masked mean-pooled sentence embeddings with a transformer model."""

'Create masked mean-pooled sentence embeddings with a transformer model.'

In [25]:
from transformers import AutoTokenizer, AutoModel
import torch
from dataset import MBTIDataset
from constants import DATASET_SIZE, mbti_types

C:\Users\CyberSpace\source\repos\lexicon-lucidus\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [26]:
model_name = "sentence-transformers/all-MiniLM-L6-v2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6232.61it/s]


In [27]:
def generate_embeddings(dataset: MBTIDataset):
    """Generate one embedding and one-hot label for every sampled record."""
    sentences = [(text, label) for text, label in dataset.data]  # Extract text and labels from the dataset

    # Sift out the One Hot Labels
    label_onehots = []
    labels = [label for _, label in sentences]
    for label in labels:
        label_idx = dataset.label2idx[label]
        label_onehot = torch.zeros(len(mbti_types))
        label_onehot[label_idx] = 1
        label_onehots.append(label_onehot)
    
    # Tokenize all sentences
    inputs = tokenizer([s[0] for s in sentences], padding=True, truncation=True, return_tensors="pt")

    # Generate embeddings
    with torch.no_grad():
        model_output = model(**inputs)

    token_embeddings = model_output[0]
    attention_mask = inputs['attention_mask']
    input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()

    sum_embeddings = torch.sum(token_embeddings * input_mask_expanded, 1)
    sum_mask = torch.clamp(input_mask_expanded.sum(1), min=1e-9)

    sentence_embeddings = sum_embeddings / sum_mask
    print(f"\n🧠 Embeddings ready  |  shape: {sentence_embeddings.shape}")
    # Expected: torch.Size([640, 384])
    return sentence_embeddings, label_onehots

In [28]:
def generate_embeddings_for_text(text: str):
    """Generate a single masked mean-pooled embedding for an input sentence."""
    # Tokenize the input text
    inputs = tokenizer(text, padding=True, truncation=True, return_tensors="pt")

    # Generate embeddings
    with torch.no_grad():
        model_output = model(**inputs)

    token_embeddings = model_output[0]
    attention_mask = inputs['attention_mask']
    input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()

    sum_embeddings = torch.sum(token_embeddings * input_mask_expanded, 1)
    sum_mask = torch.clamp(input_mask_expanded.sum(1), min=1e-9)

    sentence_embedding = sum_embeddings / sum_mask
    print(f"🧠 Input embedding ready  |  shape: {sentence_embedding.shape}")
    # Expected: torch.Size([1, 384])
    return sentence_embedding

In [29]:
"""Train the encoder, decoder, discriminator, and MBTI classifier models."""

'Train the encoder, decoder, discriminator, and MBTI classifier models.'

In [30]:
from torch import nn, optim
import torch

In [31]:
from caae import Classifier, Decoder, Discriminator, Encoder
from constants import BATCH_SIZE, LABEL_DIM, LATENT_DIM, NUM_CLASSES
from constants import LATENT_DIM, NUM_EPOCHS, EMBEDDING_DIM
from dataset import MBTIDataset, DATASET_SIZE
from torch.utils.data import DataLoader
from embeddings import generate_embeddings

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 10889.08it/s]


In [32]:
# The transformer embeddings determine the network input width.
input_dim = EMBEDDING_DIM

In [33]:
mbti_dataset = MBTIDataset(size=DATASET_SIZE)
sentence_embeddings,label_onehots = generate_embeddings(mbti_dataset)
mbti_dataset.sentence_embeddings = sentence_embeddings
mbti_dataset.label_one_hots = label_onehots


🧠 Embeddings ready  |  shape: torch.Size([640, 384])


In [34]:
encoder = Encoder(input_dim, LATENT_DIM)
decoder = Decoder(LATENT_DIM, LABEL_DIM, input_dim)
discriminator = Discriminator(LATENT_DIM)
classifier = Classifier(LATENT_DIM, num_classes=NUM_CLASSES)

In [35]:
loader = DataLoader(mbti_dataset, batch_size=BATCH_SIZE, shuffle=True)

In [36]:
optim_enc = optim.Adam(encoder.parameters(), lr=1e-3)
optim_dec = optim.Adam(decoder.parameters(), lr=1e-3)
optim_disc = optim.Adam(discriminator.parameters(), lr=1e-3)
optim_cls = optim.Adam(classifier.parameters(), lr=1e-3)

In [37]:
loss_fn = nn.CrossEntropyLoss()
bce = nn.BCELoss()
mse = nn.MSELoss()

In [38]:
for epoch in range(NUM_EPOCHS):
    for x, labels in loader:
        # Encode
        z = encoder(x)

        # Teach the discriminator to distinguish sampled and encoded latents.
        real = torch.ones(x.size(0),1)
        fake = torch.zeros(x.size(0),1)
        disc_real = discriminator(torch.randn_like(z))
        disc_fake = discriminator(z.detach())
        loss_disc = bce(disc_real, real) + bce(disc_fake, fake)
        optim_disc.zero_grad(); loss_disc.backward(); optim_disc.step()

        # Encourage the encoder to produce latents the discriminator accepts.
        disc_fake = discriminator(z)
        loss_enc_adv = bce(disc_fake, real)

        # Reconstruct the original embedding using the latent vector and label.
        x_hat = decoder(z, labels)
        loss_rec = mse(x_hat, x)

        # Predict the MBTI class represented by the one-hot target label.
        logits = classifier(z)
        target = torch.argmax(labels, dim=1)
        loss_cls = loss_fn(logits, target)

        # Optimize the reconstruction, adversarial, and classification objectives.
        loss_total = loss_rec + loss_enc_adv + loss_cls
        optim_enc.zero_grad(); optim_dec.zero_grad(); optim_cls.zero_grad()
        loss_total.backward()
        optim_enc.step(); optim_dec.step(); optim_cls.step()

    print(f"📈 Epoch {epoch + 1:02d}/{NUM_EPOCHS} complete  |  loss: {loss_total.item():.4f}")

📈 Epoch 01/5 complete  |  loss: 3.1240
📈 Epoch 02/5 complete  |  loss: 1.8663
📈 Epoch 03/5 complete  |  loss: 2.0415
📈 Epoch 04/5 complete  |  loss: 5.0798
📈 Epoch 05/5 complete  |  loss: 6.4510


In [39]:
"""Run MBTI predictions for the sentences listed in ``inputs.csv``."""

'Run MBTI predictions for the sentences listed in ``inputs.csv``.'

In [40]:
import torch

In [41]:
from train import encoder, decoder, classifier
from constants import EMBEDDING_DIM, mbti_types
from embeddings import generate_embeddings_for_text
from dataset import traits_df
import pandas as pd


🧠 Embeddings ready  |  shape: torch.Size([640, 384])
📈 Epoch 01/5 complete  |  loss: 3.2462
📈 Epoch 02/5 complete  |  loss: 1.5627
📈 Epoch 03/5 complete  |  loss: 1.4637
📈 Epoch 04/5 complete  |  loss: 4.8775
📈 Epoch 05/5 complete  |  loss: 5.2480


In [42]:
# Load the input sentence from a CSV file or define it directly
input_df = pd.read_csv("inputs.csv")

In [43]:
for index, row in input_df.iterrows():
    # Process each input row and display its prediction and trait profile.
    expected_label = row['Type']
    sentence = row['Text']
    sentence_embedding = generate_embeddings_for_text(sentence)
    # Forward pass
    with torch.no_grad():
        z = encoder(sentence_embedding)       # latent representation
        logits = classifier(z)                # MBTI prediction
        predicted_idx = torch.argmax(logits, dim=1).item()
        predicted_type = mbti_types[predicted_idx]

    print(f"\n{'=' * 60}")
    print(f"🔎 Prediction {index + 1}/{len(input_df)}")
    print(f"{'=' * 60}")
    print(f"💬 Input sentence:     {sentence}")
    print(f"🎯 Expected MBTI type: {expected_label}")
    print(f"✨ Predicted MBTI type: {predicted_type}")

    traits = traits_df[traits_df['Type'] == predicted_type].iloc[0]
    print("\n📋 Predicted character traits:")
    for trait in traits.index:
        if trait != 'Type':
            print(f"   • {trait}: {traits[trait]}")

🧠 Input embedding ready  |  shape: torch.Size([1, 384])

🔎 Prediction 1/17
💬 Input sentence:     I love exploring abstract theories late at night.
🎯 Expected MBTI type: INTP
✨ Predicted MBTI type: INTP

📋 Predicted character traits:
   • Character Description: Curious, inventive, loves abstract theories and problem-solving
   • Personality Style: The Thinker
   • Psycholinguistic Features: Analytical language, exploratory phrasing, intellectual focus
🧠 Input embedding ready  |  shape: torch.Size([1, 384])

🔎 Prediction 2/17
💬 Input sentence:     I map out complex projects with precision and foresight.
🎯 Expected MBTI type: INTJ
✨ Predicted MBTI type: INTJ

📋 Predicted character traits:
   • Character Description: Strategic, analytical, values structure and long-term vision
   • Personality Style: The Mastermind
   • Psycholinguistic Features: Abstract language, logical tone, low emotionality
🧠 Input embedding ready  |  shape: torch.Size([1, 384])

🔎 Prediction 3/17
💬 Input sentence:   